In [36]:
import pandas as pd
from io import StringIO

# read text file and only keep trip records
with open("trips.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("T;")]

# parse the trip records as pandas df
df_trips = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_trips = df_trips.iloc[:, [0,1,2,5,6,7,8,-3]]

df_trips.columns = [
    "record_type",
    "line_number",
    "trip_number",
    # "trip_direction",
    # "variant_code",
    "from_stop",
    "start_time_min",
    "end_time_min",
    "to_stop",
    "distance_km"
]

df_trips["distance_km"] = pd.to_numeric(df_trips["distance_km"], errors="coerce")
# df_trips.head()
print(df_trips)

    record_type line_number  trip_number from_stop  start_time_min  \
0             T        u001         1001    utrvec             350   
1             T        u001         1003    utrvec             364   
2             T        u001         1005    utrvec             377   
3             T        u001         1007    utrvec             389   
4             T        u001         1009    utrvec             403   
..          ...         ...          ...       ...             ...   
188           T        u001         1186    utrijs            1410   
189           T        u001         1188    utrijs            1425   
190           T        u001         1190    utrijs            1440   
191           T        u001         1192    utrijs            1455   
192           T        u001         1194    utrijs            1485   

     end_time_min to_stop  distance_km  
0             389  utrijs       12.983  
1             404  utrijs       12.983  
2             419  utrijs       12.9

In [3]:
# read text file and only keep deadhead records

with open("dhd.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("D;")]

# parse the deadhead records as pandas df
df_deadheads = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_deadheads = df_deadheads.iloc[:, [0,1,2,3,4,5,6]]

df_deadheads.columns = [
    "record_type",
    "from_stop-to_stop",
    "time_ver_0",
    "time_ver_1",
    "time_ver_2",
    "time_ver_3",
    "distance_km",
]
df_deadheads["distance_km"] = pd.to_numeric(df_deadheads["distance_km"], errors="coerce")
df_deadheads[["from_stop", "to_stop"]] = df_deadheads["from_stop-to_stop"].str.split("-", n=1, expand=True)
df_deadheads.head()

,record_type,from_stop-to_stop,time_ver_0,time_ver_1,time_ver_2,time_ver_3,distance_km,from_stop,to_stop
0,D,amdjwp-nwggam,26,26,26,26,20.0,amdjwp,nwggam
1,D,amdpri-nwggam,27,27,27,27,20.9,amdpri,nwggam
2,D,amdpri-sllgar,25,34,25,25,19.0,amdpri,sllgar
3,D,amdpri-utrgar,40,60,40,40,28.0,amdpri,utrgar
4,D,amdpri-wbdgar,55,65,55,55,43.0,amdpri,wbdgar


In [4]:
with open("parameters.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("U;")]

# read text file and only keep parameters records
df_params = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# parse the parameters records as pandas df
df_params = df_params.iloc[:, [0,2,4,9,10,12]]

# select only columns with useful information
df_params.columns = [
    "record_type",
    "cost_per_bus",
    # "cost_per_minute",
    "cost_per_km",
    "energy_compsumtion_kwh/km",
    "max_charge_rate_kwh/min",
    "battery_capacity_kwh"
]

df_params.head()

,record_type,cost_per_bus,cost_per_km,energy_compsumtion_kwh/km,max_charge_rate_kwh/min,battery_capacity_kwh
0,U,244.13,0.13,1.3,2.5,199.04


In [ ]:
# build deadhead dictionary
def build_deadhead_dict(df_deadheads):
    dh_dict = {}

    for row in df_deadheads.itertuples():
        dh_dict[(row.from_stop, row.to_stop)] = {
            "t0": row.time_ver_0,
            "t1": row.time_ver_1,
            "t2": row.time_ver_2,
            "t3": row.time_ver_3,
            "distance_km": row.distance_km
        }

    return dh_dict

# check in which time interval deadheads fall into
def get_deadhead_time(time_min, dh):
    if  331 <= time_min <= 419 or 541 <= time_min <= 899 or 1141 <= time_min <= 1439 or 1771 <= time_min <= 1859 or 1981 <= time_min <= 1999:
        return dh["t0"]
    elif 420 <= time_min <= 540 or 900 <= time_min <= 1140 or 1860 <= time_min <= 1980:
        return dh["t1"]
    elif 0 <= time_min <= 330 or 1440 <= time_min <= 1770:
        return dh["t2"]
    else:
        return dh["t3"]

# calculate all feasible deadheads
def feasible_arcs(trips, deadhead_dict, depot_stop= "utrgar", max_wait = 240):
    arcs = []
    trips_list = list(trips.itertuples())

    # feasible arcs depot -> trips
    for j in trips_list:
        if depot_stop == j.from_stop:
            travel_time = 0
        else:
            key = (depot_stop, j.from_stop)
            if key not in deadhead_dict:
                continue

            dh = deadhead_dict[key]
            travel_time = dh["t1"] # for the DH from depot to first trip take time with highest value
            distance_km = dh["distance_km"]
        
        arcs.append({
            "arc_type": "pull_out",
            "from_stop": "DEPOT",
            "to_stop": j.trip_number,
            "travel_time": travel_time,
            "distance_km" : distance_km,
            "slack": None
        })

    # feasible arcs trips -> trips
    for i in trips_list:
        for j in trips_list:
            if i.trip_number == j.trip_number:
                continue
            
            if i.to_stop == j.from_stop:
                travel_time = 0
            else:
                key = (i.to_stop, j.from_stop)
                
                if key not in deadhead_dict:
                    continue

                dh = deadhead_dict[key]
                travel_time = get_deadhead_time[i.end_time_min, dh]
                distance_km = dh["distance_km"]

            slack = j.start_time_min - (i.end_time_min + travel_time)

            if 0 <= slack <= max_wait:
                arcs.append({
                    "arc_type": "tripDH",
                    "from_stop": i.trip_number,
                    "to_stop": j.trip_number,
                    "travel_time": travel_time,
                    "distance_km" : distance_km,
                    "slack": slack
                })

    # feasible arcs trips -> depot
    for i in trips_list:
        if i.to_stop == depot_stop:
            travel_time = 0
        else:
            key = (i.to_stop, depot_stop)
            if key not in deadhead_dict:
                continue
            dh_row = deadhead_dict[key]
            travel_time = get_deadhead_time(i.end_time_min, dh_row)
            distance_km = dh["distance_km"]

        arcs.append({
            "arc_type": "pull_in",
            "from_stop": i.trip_number,
            "to_stop": "DEPOT",
            "travel_time": travel_time,
            "distance_km" : distance_km,
            "slack": None
        })
        
    return arcs

deadhead_dict = build_deadhead_dict(df_deadheads)
        
arcs = feasible_arcs(df_trips, deadhead_dict)


pull_out
